In [ ]:
import tf2onnx
from tensorflow.keras import Sequential, layers, activations, metrics, losses, models
import tensorflow as tf
import os
from onnxruntime.quantization import quantize_dynamic, QuantType
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

In [ ]:
X, y = make_regression(
    n_samples=10000,
    n_features=10,
    n_targets=1,
    n_informative=6,
    noise=0.3,
    random_state=0
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [ ]:
model = Sequential([
    layers.InputLayer(shape=(10, ), name='input'),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation=activations.linear)
])

In [ ]:
model.compile(
    optimizer='adam',
    loss=losses.MeanSquaredError(),
    metrics=[
        metrics.RootMeanSquaredError,
        metrics.R2Score,
        metrics.MeanAbsoluteError
    ]
)

In [ ]:
model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=16,
    validation_data=(X_test, y_test),
    verbose=1
)

In [33]:
model_name = 'regression_model.h5'
models.save_model(model, model_name)

In [34]:
loaded_keras_model = models.load_model(model_name)

In [35]:
onnx_model, _ = tf2onnx.convert.from_keras(
    loaded_keras_model
) # TODO: this is not working due incompatibility

KeyError: 'keras_tensor_18'